# Importar ficheiros INF001 (Informe horas entrega) para SQLite

Notebook unico com todo o processo:

1. Criar a base de dados e as tabelas `entregas_2025` / `entregas_2026`
2. Ler os ficheiros `.xls` da pasta `entregas`
3. Inserir os dados, sem duplicar linhas

Basta correr todas as celulas por ordem (Kernel -> Run All), ou uma a uma.
Podes correr este notebook sempre que houver ficheiros novos na pasta -
os dados ja importados nao serao duplicados.
\n\n**Atualizacao:** a leitura dos ficheiros INF001 (entregas) agora percorre TODAS as folhas (Worksheets) de cada ficheiro `.xls`, nao so a primeira. Se um ficheiro tiver mais do que uma folha, o notebook avisa no output e importa os dados de todas elas.

## Passo 1 — Criar a base de dados e a tabela

In [1]:
# ==========================
# 1. Definir DB_PATH e a pasta de input conforme o sistema operativo
# ==========================
import platform

if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
    PASTA_FICHEIROS = r"C:\Users\LISARR\Downloads\entregas"
elif platform.system() == 'Darwin':
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
    PASTA_FICHEIROS = "/Volumes/RR/DB/entregas"
else:
    DB_PATH = "inform_27.db"
    PASTA_FICHEIROS = "entregas"

print("DB_PATH:", DB_PATH)
print("PASTA_FICHEIROS:", PASTA_FICHEIROS)


DB_PATH: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db
PASTA_FICHEIROS: C:\Users\LISARR\Downloads\entregas


In [2]:
# ==========================
# 2. Criar a base de dados (garante que a pasta existe)
# ==========================
import sqlite3
import os

pasta = os.path.dirname(DB_PATH)
if pasta and not os.path.exists(pasta):
    os.makedirs(pasta, exist_ok=True)
    print(f"Pasta criada: {pasta}")

con = sqlite3.connect(DB_PATH)
con.close()
print(f"Base de dados criada/aberta em: {DB_PATH}")

Base de dados criada/aberta em: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db


In [3]:
# ==========================
# 3. Criar as tabelas entregas_2025 e entregas_2026
#    com os campos do ficheiro modelo INF001
# ==========================
import sqlite3

# Colunas do ficheiro INF001: nome_sql -> nome original da coluna no Excel
# (os nomes sql nao tem acentos/espacos para serem seguros em SQL)
COLUNAS_ENTREGAS = {
    "PROPIETARIO_DT": "Propietario DT",
    "RECORRIDO_CAMION": "Recorrido Camión",
    "DESTINO": "Destino",
    "HORA_TEORICA_LLEGADA": "Hora teórica Llegada",
    "HORA_REAL_LLEGADA": "Hora Real Llegada",
    "HORA_SALIDA_DESTINO": "Hora Salida de Destino",
    "RETRASO_LLEGADA": "Retraso Llegada",
    "TIEMPO_ESPERA": "Tiempo de Espera",
    "NUM_UT": "Núm. UT",
    "FECHA_TEORICA_LLEGADA": "Fecha Teórica Llegada",
    "FECHA_REAL_LLEGADA": "Fecha Real Llegada",
    "TEMPERATURA": "Temperatura",
    "EMPRESA_TRANSPORTE": "Empresa de Transporte",
    "MATRICULA_REMOLQUE": "Matrícula Remolque",
    "MOTIVO": "Motivo",
    "OBSERVACIONES": "Observaciones",
}

COLUNA_ORIGEM = "ficheiro_origem"  # coluna extra: de que ficheiro veio a linha

colunas_sql = ",\n    ".join(f'"{c}" TEXT' for c in COLUNAS_ENTREGAS)

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

for ano in ("2025", "2026"):
    cur.execute(f'''
        CREATE TABLE IF NOT EXISTS entregas_{ano} (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            {colunas_sql},
            "{COLUNA_ORIGEM}" TEXT
        )
    ''')
    print(f"Tabela entregas_{ano} criada (ou ja existia).")

con.commit()
con.close()
print("OK - tabelas prontas.")


Tabela entregas_2025 criada (ou ja existia).
Tabela entregas_2026 criada (ou ja existia).
OK - tabelas prontas.


In [4]:
# ==========================
# 4. Confirmar: listar as tabelas e colunas criadas
# ==========================
import sqlite3

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("Tabelas na base de dados:")
for (nome,) in cur.fetchall():
    print(" -", nome)

print()
for tabela in ("entregas_2025", "entregas_2026"):
    cur.execute(f"PRAGMA table_info({tabela})")
    colunas = cur.fetchall()
    print(f"Colunas de {tabela}: {len(colunas)}")
    for c in colunas:
        print("   ", c[1], c[2])
    print()

con.close()


Tabelas na base de dados:
 - dados_2025
 - sqlite_sequence
 - dados_2026
 - entregas_2025
 - entregas_2026
 - cargas_2025
 - cargas_2026

Colunas de entregas_2025: 18
    id INTEGER
    PROPIETARIO_DT TEXT
    RECORRIDO_CAMION TEXT
    DESTINO TEXT
    HORA_TEORICA_LLEGADA TEXT
    HORA_REAL_LLEGADA TEXT
    HORA_SALIDA_DESTINO TEXT
    RETRASO_LLEGADA TEXT
    TIEMPO_ESPERA TEXT
    NUM_UT TEXT
    FECHA_TEORICA_LLEGADA TEXT
    FECHA_REAL_LLEGADA TEXT
    TEMPERATURA TEXT
    EMPRESA_TRANSPORTE TEXT
    MATRICULA_REMOLQUE TEXT
    MOTIVO TEXT
    OBSERVACIONES TEXT
    ficheiro_origem TEXT

Colunas de entregas_2026: 18
    id INTEGER
    PROPIETARIO_DT TEXT
    RECORRIDO_CAMION TEXT
    DESTINO TEXT
    HORA_TEORICA_LLEGADA TEXT
    HORA_REAL_LLEGADA TEXT
    HORA_SALIDA_DESTINO TEXT
    RETRASO_LLEGADA TEXT
    TIEMPO_ESPERA TEXT
    NUM_UT TEXT
    FECHA_TEORICA_LLEGADA TEXT
    FECHA_REAL_LLEGADA TEXT
    TEMPERATURA TEXT
    EMPRESA_TRANSPORTE TEXT
    MATRICULA_REMOLQUE TEXT
   

## Passo 2 — Ler os ficheiros da pasta

In [5]:
# ==========================
# 1. Localizar os ficheiros .xls na pasta
# ==========================
import glob
import os

def listar_ficheiros_excel(pasta):
    """Procura ficheiros .xls (tambem em subpastas), sem duplicar por
    causa de maiusculas/minusculas."""
    encontrados = glob.glob(os.path.join(pasta, "**", "*.xls"), recursive=True)
    vistos = set()
    ficheiros = []
    for caminho in encontrados:
        chave = os.path.normcase(os.path.abspath(caminho))
        if chave not in vistos:
            vistos.add(chave)
            ficheiros.append(caminho)
    return sorted(ficheiros)

ficheiros = listar_ficheiros_excel(PASTA_FICHEIROS)

print(f"Pasta: {PASTA_FICHEIROS}")
print(f"Ficheiros .xls encontrados: {len(ficheiros)}")
for f in ficheiros[:20]:
    print(" -", os.path.basename(f))
if len(ficheiros) > 20:
    print(f"   ... e mais {len(ficheiros) - 20} ficheiros")


Pasta: C:\Users\LISARR\Downloads\entregas
Ficheiros .xls encontrados: 21
 - INF001 Informe horas entrega por DT2 (10).xls
 - INF001 Informe horas entrega por DT2 (11).xls
 - INF001 Informe horas entrega por DT2 (12).xls
 - INF001 Informe horas entrega por DT2 (13).xls
 - INF001 Informe horas entrega por DT2 (14).xls
 - INF001 Informe horas entrega por DT2 (15).xls
 - INF001 Informe horas entrega por DT2 (16).xls
 - INF001 Informe horas entrega por DT2 (17).xls
 - INF001 Informe horas entrega por DT2 (18).xls
 - INF001 Informe horas entrega por DT2 (19).xls
 - INF001 Informe horas entrega por DT2 (20).xls
 - INF001 Informe horas entrega por DT2 (21).xls
 - INF001 Informe horas entrega por DT2 (22).xls
 - INF001 Informe horas entrega por DT2 (23).xls
 - INF001 Informe horas entrega por DT2 (3).xls
 - INF001 Informe horas entrega por DT2 (4).xls
 - INF001 Informe horas entrega por DT2 (5).xls
 - INF001 Informe horas entrega por DT2 (6).xls
 - INF001 Informe horas entrega por DT2 (7).xls
 

In [6]:
# ==========================
# 2. Funcao para ler um ficheiro "Excel XML Spreadsheet 2003"
#    (o formato real destes .xls - por isso o Excel avisa que a
#    extensao nao coincide, mas nao estao corrompidos)
#
#    Os ficheiros INF001 tem 1-2 linhas de titulo antes do cabecalho
#    real, por isso a funcao avanca ate encontrar a 1a linha com varias
#    colunas preenchidas - essa e considerada o cabecalho.
#
#    IMPORTANTE: um ficheiro Excel pode ter mais do que uma folha
#    (Worksheet). Esta funcao agora percorre TODAS as folhas do
#    ficheiro e devolve uma lista com uma entrada por folha:
#        [(nome_folha, cabecalho, linhas), (nome_folha, cabecalho, linhas), ...]
#    Antes, so a primeira folha era lida (root.find so devolve a 1a
#    ocorrencia) e os dados de folhas extra eram ignorados em silencio.
# ==========================
import xml.etree.ElementTree as ET

NS = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}
SS_INDEX = "{urn:schemas-microsoft-com:office:spreadsheet}Index"
SS_NAME = "{urn:schemas-microsoft-com:office:spreadsheet}Name"

def ler_linhas_xml_spreadsheet(caminho_ficheiro, min_colunas_cabecalho=5):
    """
    Le um ficheiro no formato 'Excel XML Spreadsheet 2003' e devolve
    uma lista com uma entrada (nome_folha, cabecalho, linhas) por CADA
    folha (Worksheet) encontrada no ficheiro.

    Respeita o atributo ss:Index de cada <Cell>, porque o Excel omite
    celulas vazias e usa esse indice para indicar a posicao real da
    coluna. Salta linhas de titulo ate encontrar a linha de cabecalho
    real, em cada folha separadamente (cada folha pode ter o seu
    proprio cabecalho).
    """
    tree = ET.parse(caminho_ficheiro)
    root = tree.getroot()

    worksheets = root.findall("ss:Worksheet", NS)
    if not worksheets:
        raise ValueError("Nao foi encontrada nenhuma 'Worksheet' no ficheiro.")

    def extrair_linha(linha_xml):
        valores = []
        proximo_indice = 1
        for cell in linha_xml.findall("ss:Cell", NS):
            idx = cell.get(SS_INDEX)
            idx = int(idx) if idx is not None else proximo_indice
            while len(valores) < idx - 1:
                valores.append(None)
            data_el = cell.find("ss:Data", NS)
            valor = data_el.text if data_el is not None else None
            valores.append(valor)
            proximo_indice = idx + 1
        return valores

    resultado = []
    for worksheet in worksheets:
        nome_folha = worksheet.get(SS_NAME, "")

        tabela = worksheet.find("ss:Table", NS)
        if tabela is None:
            continue

        linhas_xml = tabela.findall("ss:Row", NS)
        if not linhas_xml:
            continue

        idx_cabecalho = None
        cabecalho = None
        for i, linha_xml in enumerate(linhas_xml):
            valores = extrair_linha(linha_xml)
            if len([v for v in valores if v not in (None, "")]) >= min_colunas_cabecalho:
                idx_cabecalho = i
                cabecalho = valores
                break

        if idx_cabecalho is None:
            # esta folha nao tem uma linha de cabecalho identificavel
            # (ex: folha vazia ou so com notas) - ignora-se so esta folha
            continue

        cabecalho = [c.strip() if c else f"COLUNA_{i+1}" for i, c in enumerate(cabecalho)]

        linhas = []
        n_colunas = len(cabecalho)
        for linha_xml in linhas_xml[idx_cabecalho + 1:]:
            valores = extrair_linha(linha_xml)
            if len(valores) < n_colunas:
                valores += [None] * (n_colunas - len(valores))
            elif len(valores) > n_colunas:
                valores = valores[:n_colunas]
            linhas.append(valores)

        resultado.append((nome_folha, cabecalho, linhas))

    return resultado

print("Funcao ler_linhas_xml_spreadsheet() pronta (le todas as folhas do ficheiro).")


Funcao ler_linhas_xml_spreadsheet() pronta.


## Passo 3 — Inserir os dados (sem duplicar)

In [7]:
# ==========================
# 1. Criar indice UNICO para evitar duplicar a mesma linha
#
#    Este relatorio nao tem um ID unico tipo CODEDT, por isso a chave
#    usada e (Num. UT + Destino + Hora teorica Llegada + Fecha teorica
#    Llegada). Usamos COALESCE(..., '') porque o SQLite trata cada
#    valor NULL como diferente de outro NULL - sem isto, linhas com
#    Destino/Num.UT em branco nunca seriam consideradas duplicadas.
# ==========================
import sqlite3

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

for ano in ("2025", "2026"):
    cur.execute(
        f'CREATE UNIQUE INDEX IF NOT EXISTS idx_entregas_{ano}_chave '
        f'ON entregas_{ano}(COALESCE("NUM_UT", \'\'), COALESCE("DESTINO", \'\'), '
        f'COALESCE("HORA_TEORICA_LLEGADA", \'\'), COALESCE("FECHA_TEORICA_LLEGADA", \'\'))'
    )

con.commit()
con.close()
print("Indices unicos prontos em entregas_2025 e entregas_2026.")


Indices unicos prontos em entregas_2025 e entregas_2026.


In [8]:
# ==========================
# 2. Preparar as instrucoes de insercao (INSERT OR IGNORE)
# ==========================
cols_sql = ", ".join(f'"{c}"' for c in COLUNAS_ENTREGAS)
placeholders = ", ".join(["?"] * len(COLUNAS_ENTREGAS))

sql_2025 = (
    f'INSERT OR IGNORE INTO entregas_2025 ({cols_sql}, "{COLUNA_ORIGEM}") '
    f'VALUES ({placeholders}, ?)'
)
sql_2026 = (
    f'INSERT OR IGNORE INTO entregas_2026 ({cols_sql}, "{COLUNA_ORIGEM}") '
    f'VALUES ({placeholders}, ?)'
)

print("Instrucoes SQL prontas.")


Instrucoes SQL prontas.


In [9]:
# ==========================
# 3. Loop principal: ler cada ficheiro (todas as folhas) e inserir na
#    tabela do ano certo
#
#    O ano e determinado pelo campo "Fecha Teorica Llegada" (formato
#    DD/MM/AAAA).
# ==========================
import sqlite3
from datetime import datetime

def extrair_ano(mapa):
    """Devolve o ano (string, 4 digitos) a partir de Fecha Teorica Llegada,
    ou None se nao houver data valida."""
    valor = mapa.get(COLUNAS_ENTREGAS["FECHA_TEORICA_LLEGADA"])
    if valor and str(valor).strip().upper() != "N/D":
        partes = str(valor).strip().split("/")
        if len(partes) == 3 and partes[2].isdigit():
            return partes[2]
    return None

total_inseridos_2025 = 0
total_inseridos_2026 = 0
total_duplicados = 0
total_sem_data = 0

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

for i, caminho in enumerate(ficheiros, start=1):
    nome_ficheiro = os.path.basename(caminho)

    try:
        folhas = ler_linhas_xml_spreadsheet(caminho)
    except Exception as e:
        print(f"[{i}/{len(ficheiros)}] [ERRO] Falha a ler '{nome_ficheiro}': {e}")
        continue

    if len(folhas) > 1:
        nomes_folhas = [nome for nome, _, _ in folhas]
        print(f"[{i}/{len(ficheiros)}] A processar: {nome_ficheiro} "
              f"({len(folhas)} folhas encontradas: {nomes_folhas})")
    else:
        print(f"[{i}/{len(ficheiros)}] A processar: {nome_ficheiro}")

    batch_2025, batch_2026 = [], []
    sem_data = 0

    for nome_folha, cabecalho, linhas in folhas:
        for valores in linhas:
            if all(v is None or str(v).strip() == "" for v in valores):
                continue

            mapa = dict(zip(cabecalho, valores))
            valores_ordenados = [mapa.get(nome_original) for nome_original in COLUNAS_ENTREGAS.values()]

            ano = extrair_ano(mapa)
            if ano == "2025":
                batch_2025.append(valores_ordenados + [nome_ficheiro])
            elif ano == "2026":
                batch_2026.append(valores_ordenados + [nome_ficheiro])
            else:
                sem_data += 1

    inseridos_2025 = inseridos_2026 = 0

    if batch_2025:
        antes = con.total_changes
        cur.executemany(sql_2025, batch_2025)
        con.commit()
        inseridos_2025 = con.total_changes - antes

    if batch_2026:
        antes = con.total_changes
        cur.executemany(sql_2026, batch_2026)
        con.commit()
        inseridos_2026 = con.total_changes - antes

    duplicados = (len(batch_2025) - inseridos_2025) + (len(batch_2026) - inseridos_2026)

    agora = datetime.now().isoformat(timespec="seconds")
    print(f"  -> {inseridos_2025} novas em 2025, {inseridos_2026} novas em 2026, "
          f"{duplicados} ja existiam (chave repetida), {sem_data} sem data valida "
          f"| importado em: {agora}")

    total_inseridos_2025 += inseridos_2025
    total_inseridos_2026 += inseridos_2026
    total_duplicados += duplicados
    total_sem_data += sem_data

con.close()

print("=" * 70)
print("RESUMO FINAL")
print("=" * 70)
print(f"Ficheiros processados:        {len(ficheiros)}")
print(f"Linhas novas inseridas 2025:  {total_inseridos_2025}")
print(f"Linhas novas inseridas 2026:  {total_inseridos_2026}")
print(f"Linhas ja existentes (duplicadas): {total_duplicados}")
print(f"Linhas sem data valida:       {total_sem_data}")
print("=" * 70)


[1/21] A processar: INF001 Informe horas entrega por DT2 (10).xls
  -> 0 novas em 2025, 0 novas em 2026, 4959 ja existiam (chave repetida), 1 sem data valida | importado em: 2026-07-22T13:53:48
[2/21] A processar: INF001 Informe horas entrega por DT2 (11).xls
  -> 0 novas em 2025, 0 novas em 2026, 7652 ja existiam (chave repetida), 1 sem data valida | importado em: 2026-07-22T13:53:49
[3/21] A processar: INF001 Informe horas entrega por DT2 (12).xls
  -> 0 novas em 2025, 0 novas em 2026, 5 ja existiam (chave repetida), 1 sem data valida | importado em: 2026-07-22T13:53:49
[4/21] A processar: INF001 Informe horas entrega por DT2 (13).xls
  -> 0 novas em 2025, 0 novas em 2026, 4902 ja existiam (chave repetida), 1 sem data valida | importado em: 2026-07-22T13:53:50
[5/21] A processar: INF001 Informe horas entrega por DT2 (14).xls
  -> 0 novas em 2025, 0 novas em 2026, 7317 ja existiam (chave repetida), 1 sem data valida | importado em: 2026-07-22T13:53:51
[6/21] A processar: INF001 Inform

## Passo 4 — Importar ficheiros CRONO para a tabela `cargas`

Mesma lógica dos passos 1-3, mas para um terceiro relatório:

- Lido da pasta `PASTA_FICHEIROS_CARGAS` (`C:\Users\LISARR\Downloads\crono`)
- Gravado na tabela `cargas_2025` / `cargas_2026`
- Na **mesma base de dados** (`DB_PATH`)

**Diferença importante:** ao contrário do ficheiro INF001 (XML "Excel
Spreadsheet 2003"), o ficheiro CRONO é texto separado por tabs (TSV), com
acentuação em ISO-8859-1 — por isso a leitura usa `csv.reader` em vez do
parser de XML.

O cabeçalho tem 2 colunas chamadas `Temperatura` (uma é a temperatura
exigida, a outra a medida). Por terem o mesmo nome, não é seguro construir
um dicionário nome→valor; por isso a leitura confia na **posição/ordem**
das colunas, validando primeiro que o cabeçalho do ficheiro é exatamente o
esperado.

**Chave de deduplicação:** `Ruta` sozinho não é único (há linhas repetidas
com o mesmo número de rota, mas dados diferentes), por isso a chave usada
é a combinação (`Ruta` + `Punto Suministro` + `Base` + `Tractora` +
`Fecha Prevista Posicionamiento` + `Hora Prevista Posicionamiento`).


In [10]:
# ==========================
# 1. Definir a pasta de input dos ficheiros CRONO (cargas)
# ==========================
import platform

if platform.system() == 'Windows':
    PASTA_FICHEIROS_CARGAS = r"C:\Users\LISARR\Downloads\crono"
elif platform.system() == 'Darwin':
    PASTA_FICHEIROS_CARGAS = "/Volumes/RR/DB/crono"
else:
    PASTA_FICHEIROS_CARGAS = "crono"

print("PASTA_FICHEIROS_CARGAS:", PASTA_FICHEIROS_CARGAS)


PASTA_FICHEIROS_CARGAS: C:\Users\LISARR\Downloads\crono


In [11]:
# ==========================
# 2. Criar as tabelas cargas_2025 e cargas_2026
#    com os campos do ficheiro modelo CRONO
# ==========================
import sqlite3

# Colunas do ficheiro CRONO: nome_sql -> nome original da coluna
# (o ficheiro tem 2 colunas chamadas "Temperatura"; a 2a fica
# TEMPERATURA_MEDIDA1, distinta de TEMPERATURA_REQUERIDA)
COLUNAS_CARGAS = {
    "TIPO_SUMINISTRO": "Tipo Suministro",
    "BASE": "Base",
    "LANZADERA": "Lanzadera",
    "PUNTO_SUMINISTRO": "Punto Suministro",
    "RUTA": "Ruta",
    "PALES_TMS": "Palés TMS",
    "TEMPERATURA_REQUERIDA": "Temperatura",
    "AGENCIA": "Agencia",
    "TRANSPORTISTA": "Transportista",
    "DNI": "DNI",
    "PRECINTO": "Precinto",
    "TELEFONO": "Teléfono",
    "TRACTORA": "Tractora",
    "REMOLQUE": "Remolque",
    "FECHA_PREVISTA_POSICIONAMIENTO": "Fecha Prevista Posicionamiento",
    "HORA_PREVISTA_POSICIONAMIENTO": "Hora Prevista Posicionamiento",
    "FECHA_REAL_POSICIONAMIENTO": "Fecha Real Posicionamiento",
    "HORA_REAL_POSICIONAMIENTO": "Hora Real Posicionamiento",
    "FECHA_REAL_ENTRADA": "Fecha Real Entrada",
    "HORA_REAL_ENTRADA": "Hora Real Entrada",
    "MUELLE": "Muelle",
    "FECHA_PREVISTA_SALIDA": "Fecha Prevista Salida",
    "HORA_PREVISTA_SALIDA": "Hora Prevista Salida",
    "FECHA_REAL_SALIDA": "Fecha Real Salida",
    "HORA_REAL_SALIDA": "Hora Real Salida",
    "FECHA_PREVISTA_ENTREGA": "Fecha Prevista Entrega",
    "HORA_PREVISTA_ENTREGA": "Hora Prevista Entrega",
    "FECHA_REAL_ENTREGA": "Fecha Real Entrega",
    "HORA_REAL_ENTREGA": "Hora Real Entrega",
    "HORA_SALIDA_ENTREGA": "Hora Salida Entrega",
    "OBSERVACIONES": "Observaciones",
    "COMENTARIOS": "Comentarios",
    "ZONA": "Zona",
    "CLIENTE": "Cliente",
    "AUTORIZADO_AUTOCARGA": "Autorizado autocarga",
    "AUTOCARGA": "Autocarga",
    "HUECOS_TMS": "Huecos TMS",
    "HUECOS_CARGA": "Huecos carga",
    "HUECOS_DESCARGA": "Huecos descarga",
    "TEMPERATURA_MEDIDA1": "Temperatura",
    "TEMPERATURA_MEDIDA2": "Temperatura2",
    "MOTIVO": "Motivo",
    "ESTADO": "Estado",
    "ESTADO_MERCANCIA": "Estado mercancía",
    "ESTADO_CAJA": "Estado caja",
    "ESTADO_OLORES": "Estado olores",
    "ESTADO_LIMPIEZA_VEHICULO": "Estado limpieza vehículo",
    "ESTADO_VEHICULO_SECO": "Estado vehículo seco",
    "ESTADO_LIBRE_PLAGAS": "Estado libre de plagas",
}

# Ordem exata das colunas no ficheiro (usada para validar o cabecalho
# antes de ler os dados por posicao)
ORDEM_CABECALHO_CARGAS = list(COLUNAS_CARGAS.values())

COLUNA_ORIGEM_CARGAS = "ficheiro_origem"

colunas_sql_cargas = ",\n    ".join(f'"{c}" TEXT' for c in COLUNAS_CARGAS)

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

for ano in ("2025", "2026"):
    cur.execute(f'''
        CREATE TABLE IF NOT EXISTS cargas_{ano} (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            {colunas_sql_cargas},
            "{COLUNA_ORIGEM_CARGAS}" TEXT
        )
    ''')
    print(f"Tabela cargas_{ano} criada (ou ja existia).")

con.commit()
con.close()
print("OK - tabelas de cargas prontas, na mesma base de dados:", DB_PATH)


Tabela cargas_2025 criada (ou ja existia).
Tabela cargas_2026 criada (ou ja existia).
OK - tabelas de cargas prontas, na mesma base de dados: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db


In [12]:
# ==========================
# 3. Confirmar: listar as tabelas e colunas da base de dados
# ==========================
import sqlite3

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("Tabelas na base de dados:")
for (nome,) in cur.fetchall():
    print(" -", nome)

print()
for tabela in ("cargas_2025", "cargas_2026"):
    cur.execute(f"PRAGMA table_info({tabela})")
    colunas = cur.fetchall()
    print(f"Colunas de {tabela}: {len(colunas)}")
    for c in colunas:
        print("   ", c[1], c[2])
    print()

con.close()


Tabelas na base de dados:
 - dados_2025
 - sqlite_sequence
 - dados_2026
 - entregas_2025
 - entregas_2026
 - cargas_2025
 - cargas_2026

Colunas de cargas_2025: 51
    id INTEGER
    TIPO_SUMINISTRO TEXT
    BASE TEXT
    LANZADERA TEXT
    PUNTO_SUMINISTRO TEXT
    RUTA TEXT
    PALES_TMS TEXT
    TEMPERATURA_REQUERIDA TEXT
    AGENCIA TEXT
    TRANSPORTISTA TEXT
    DNI TEXT
    PRECINTO TEXT
    TELEFONO TEXT
    TRACTORA TEXT
    REMOLQUE TEXT
    FECHA_PREVISTA_POSICIONAMIENTO TEXT
    HORA_PREVISTA_POSICIONAMIENTO TEXT
    FECHA_REAL_POSICIONAMIENTO TEXT
    HORA_REAL_POSICIONAMIENTO TEXT
    FECHA_REAL_ENTRADA TEXT
    HORA_REAL_ENTRADA TEXT
    MUELLE TEXT
    FECHA_PREVISTA_SALIDA TEXT
    HORA_PREVISTA_SALIDA TEXT
    FECHA_REAL_SALIDA TEXT
    HORA_REAL_SALIDA TEXT
    FECHA_PREVISTA_ENTREGA TEXT
    HORA_PREVISTA_ENTREGA TEXT
    FECHA_REAL_ENTREGA TEXT
    HORA_REAL_ENTREGA TEXT
    HORA_SALIDA_ENTREGA TEXT
    OBSERVACIONES TEXT
    COMENTARIOS TEXT
    ZONA TEXT
    CLI

## Passo 5 — Ler os ficheiros CRONO da pasta `crono`

In [13]:
# ==========================
# 1. Localizar os ficheiros .xls na pasta crono
#    (reutiliza a funcao listar_ficheiros_excel ja definida no Passo 2)
# ==========================
ficheiros_cargas = listar_ficheiros_excel(PASTA_FICHEIROS_CARGAS)

print(f"Pasta: {PASTA_FICHEIROS_CARGAS}")
print(f"Ficheiros .xls encontrados: {len(ficheiros_cargas)}")
for f in ficheiros_cargas[:20]:
    print(" -", os.path.basename(f))
if len(ficheiros_cargas) > 20:
    print(f"   ... e mais {len(ficheiros_cargas) - 20} ficheiros")


Pasta: C:\Users\LISARR\Downloads\crono
Ficheiros .xls encontrados: 21
 - crono_1784655922836.xls
 - crono_1784655960761.xls
 - crono_1784655998553.xls
 - crono_1784656022355.xls
 - crono_1784656044889.xls
 - crono_1784656070233.xls
 - crono_1784656086169.xls
 - crono_1784663524006.xls
 - crono_1784663538828.xls
 - crono_1784663555788.xls
 - crono_1784663582721.xls
 - crono_1784663824862.xls
 - crono_1784724279791.xls
 - crono_1784724302359.xls
 - crono_1784724343747.xls
 - crono_1784724374173.xls
 - crono_1784724406177.xls
 - crono_1784724431556.xls
 - crono_1784724455140.xls
 - crono_1784724535468.xls
   ... e mais 1 ficheiros


In [14]:
# ==========================
# 2. Funcao para ler os ficheiros CRONO (texto separado por tabs)
#
#    Estes ficheiros NAO sao XML como o inform_27/entregas - sao texto
#    tab-separated, com acentos em ISO-8859-1. O cabecalho tem 2 colunas
#    chamadas "Temperatura", por isso NAO fazemos dict(zip(cabecalho,
#    valores)) (isso perderia a 1a ocorrencia) - em vez disso validamos
#    que o cabecalho bate certo com ORDEM_CABECALHO_CARGAS e usamos a
#    posicao das colunas diretamente.
# ==========================
import csv

def ler_linhas_csv_cargas(caminho_ficheiro):
    """
    Le um ficheiro CRONO (texto tab-separated, ISO-8859-1) e devolve
    (cabecalho, lista_de_linhas), tal como as funcoes de leitura
    anteriores.
    """
    with open(caminho_ficheiro, encoding="iso-8859-1", newline="") as f:
        reader = csv.reader(f, delimiter="\t")
        linhas_ficheiro = list(reader)

    if not linhas_ficheiro:
        return [], []

    cabecalho = linhas_ficheiro[0]
    linhas = linhas_ficheiro[1:]

    n_colunas = len(cabecalho)
    linhas_normalizadas = []
    for valores in linhas:
        if len(valores) < n_colunas:
            valores = valores + [None] * (n_colunas - len(valores))
        elif len(valores) > n_colunas:
            valores = valores[:n_colunas]
        linhas_normalizadas.append(valores)

    return cabecalho, linhas_normalizadas

print("Funcao ler_linhas_csv_cargas() pronta.")


Funcao ler_linhas_csv_cargas() pronta.


In [15]:
# ==========================
# 3. Testar a leitura em todos os ficheiros encontrados
#    (confirma que o cabecalho bate certo com o esperado)
# ==========================
for caminho in ficheiros_cargas:
    nome_ficheiro = os.path.basename(caminho)
    try:
        cabecalho, linhas = ler_linhas_csv_cargas(caminho)
    except Exception as e:
        print(f"[ERRO] {nome_ficheiro}: nao foi possivel ler -> {e}")
        continue

    cabecalho_ok = cabecalho == ORDEM_CABECALHO_CARGAS
    print(f"{nome_ficheiro}: {len(linhas)} linhas | colunas: {len(cabecalho)} | "
          f"cabecalho igual ao esperado: {cabecalho_ok}")

    if not cabecalho_ok:
        print(f"   [AVISO] cabecalho diferente do esperado - este ficheiro sera ignorado na insercao.")


crono_1784655922836.xls: 1023 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784655960761.xls: 1152 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784655998553.xls: 1113 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784656022355.xls: 1171 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784656044889.xls: 1237 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784656070233.xls: 1265 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784656086169.xls: 1171 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784663524006.xls: 1833 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784663538828.xls: 2018 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784663555788.xls: 2018 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784663582721.xls: 1849 linhas | colunas: 49 | cabecalho igual ao esperado: True
crono_1784663824862.xls: 442 linhas | colun

## Passo 6 — Inserir os dados de cargas (sem duplicar)

In [16]:
# ==========================
# 1. Criar indice UNICO para evitar duplicar a mesma linha
#
#    "Ruta" sozinho nao e unico (ha linhas com o mesmo numero de rota
#    mas dados diferentes - ex: Base ou Tractora diferentes). A chave
#    usada e (Ruta + Punto Suministro + Base + Tractora + Fecha Prevista
#    Posicionamiento + Hora Prevista Posicionamiento), testada e
#    confirmada como unica no ficheiro modelo.
# ==========================
import sqlite3

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

for ano in ("2025", "2026"):
    cur.execute(
        f'CREATE UNIQUE INDEX IF NOT EXISTS idx_cargas_{ano}_chave '
        f'ON cargas_{ano}(COALESCE("RUTA", \'\'), COALESCE("PUNTO_SUMINISTRO", \'\'), '
        f'COALESCE("BASE", \'\'), COALESCE("TRACTORA", \'\'), '
        f'COALESCE("FECHA_PREVISTA_POSICIONAMIENTO", \'\'), '
        f'COALESCE("HORA_PREVISTA_POSICIONAMIENTO", \'\'))'
    )

con.commit()
con.close()
print("Indices unicos prontos em cargas_2025 e cargas_2026.")


Indices unicos prontos em cargas_2025 e cargas_2026.


In [17]:
# ==========================
# 2. Loop principal: ler cada ficheiro e inserir na tabela do ano certo
#
#    O ano e determinado por "Fecha Real Entrega" (formato DD/MM/AAAA);
#    se vier vazia, usa-se "Fecha Prevista Entrega", e por fim
#    "Fecha Prevista Posicionamiento" (esta ultima esta sempre preenchida).
#
#    As instrucoes SQL sao preparadas aqui mesmo, para esta celula nao
#    depender de teres corrido outra celula antes (evita NameError se
#    reiniciares o kernel).
# ==========================
import sqlite3
from datetime import datetime

cols_sql_cargas = ", ".join(f'"{c}"' for c in COLUNAS_CARGAS)
placeholders_cargas = ", ".join(["?"] * len(COLUNAS_CARGAS))

sql_cargas_2025 = (
    f'INSERT OR IGNORE INTO cargas_2025 ({cols_sql_cargas}, "{COLUNA_ORIGEM_CARGAS}") '
    f'VALUES ({placeholders_cargas}, ?)'
)
sql_cargas_2026 = (
    f'INSERT OR IGNORE INTO cargas_2026 ({cols_sql_cargas}, "{COLUNA_ORIGEM_CARGAS}") '
    f'VALUES ({placeholders_cargas}, ?)'
)

idx_fecha_real_entrega = ORDEM_CABECALHO_CARGAS.index("Fecha Real Entrega")
idx_fecha_prevista_entrega = ORDEM_CABECALHO_CARGAS.index("Fecha Prevista Entrega")
idx_fecha_prevista_posicionamiento = ORDEM_CABECALHO_CARGAS.index("Fecha Prevista Posicionamiento")

def extrair_ano_cargas(valores_ordenados):
    """Devolve o ano (string, 4 digitos) a partir das datas do relatorio,
    ou None se nenhuma data valida for encontrada."""
    for idx in (idx_fecha_real_entrega, idx_fecha_prevista_entrega, idx_fecha_prevista_posicionamiento):
        valor = valores_ordenados[idx]
        if valor and str(valor).strip().upper() != "N/D":
            partes = str(valor).strip().split("/")
            if len(partes) == 3 and partes[2].isdigit():
                return partes[2]
    return None

total_inseridos_2025 = 0
total_inseridos_2026 = 0
total_duplicados = 0
total_sem_data = 0
total_cabecalho_invalido = 0

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

for i, caminho in enumerate(ficheiros_cargas, start=1):
    nome_ficheiro = os.path.basename(caminho)
    print(f"[{i}/{len(ficheiros_cargas)}] A processar: {nome_ficheiro}")

    try:
        cabecalho, linhas = ler_linhas_csv_cargas(caminho)
    except Exception as e:
        print(f"  [ERRO] Falha a ler '{nome_ficheiro}': {e}")
        continue

    if cabecalho != ORDEM_CABECALHO_CARGAS:
        print(f"  [AVISO] cabecalho de '{nome_ficheiro}' diferente do esperado. A ignorar ficheiro.")
        total_cabecalho_invalido += 1
        continue

    batch_2025, batch_2026 = [], []
    sem_data = 0

    for valores in linhas:
        if all(v is None or str(v).strip() == "" for v in valores):
            continue

        ano = extrair_ano_cargas(valores)
        if ano == "2025":
            batch_2025.append(valores + [nome_ficheiro])
        elif ano == "2026":
            batch_2026.append(valores + [nome_ficheiro])
        else:
            sem_data += 1

    inseridos_2025 = inseridos_2026 = 0

    if batch_2025:
        antes = con.total_changes
        cur.executemany(sql_cargas_2025, batch_2025)
        con.commit()
        inseridos_2025 = con.total_changes - antes

    if batch_2026:
        antes = con.total_changes
        cur.executemany(sql_cargas_2026, batch_2026)
        con.commit()
        inseridos_2026 = con.total_changes - antes

    duplicados = (len(batch_2025) - inseridos_2025) + (len(batch_2026) - inseridos_2026)

    agora = datetime.now().isoformat(timespec="seconds")
    print(f"  -> {inseridos_2025} novas em 2025, {inseridos_2026} novas em 2026, "
          f"{duplicados} ja existiam (chave repetida), {sem_data} sem data valida "
          f"| importado em: {agora}")

    total_inseridos_2025 += inseridos_2025
    total_inseridos_2026 += inseridos_2026
    total_duplicados += duplicados
    total_sem_data += sem_data

con.close()

print("=" * 70)
print("RESUMO FINAL - CARGAS (CRONO)")
print("=" * 70)
print(f"Ficheiros processados:        {len(ficheiros_cargas)}")
print(f"Ficheiros com cabecalho invalido (ignorados): {total_cabecalho_invalido}")
print(f"Linhas novas inseridas 2025:  {total_inseridos_2025}")
print(f"Linhas novas inseridas 2026:  {total_inseridos_2026}")
print(f"Linhas ja existentes (duplicadas): {total_duplicados}")
print(f"Linhas sem data valida:       {total_sem_data}")
print("=" * 70)


[1/21] A processar: crono_1784655922836.xls
  -> 0 novas em 2025, 0 novas em 2026, 1023 ja existiam (chave repetida), 0 sem data valida | importado em: 2026-07-22T13:54:02
[2/21] A processar: crono_1784655960761.xls
  -> 0 novas em 2025, 0 novas em 2026, 1152 ja existiam (chave repetida), 0 sem data valida | importado em: 2026-07-22T13:54:02
[3/21] A processar: crono_1784655998553.xls
  -> 0 novas em 2025, 0 novas em 2026, 1113 ja existiam (chave repetida), 0 sem data valida | importado em: 2026-07-22T13:54:02
[4/21] A processar: crono_1784656022355.xls
  -> 0 novas em 2025, 0 novas em 2026, 1171 ja existiam (chave repetida), 0 sem data valida | importado em: 2026-07-22T13:54:02
[5/21] A processar: crono_1784656044889.xls
  -> 0 novas em 2025, 0 novas em 2026, 1237 ja existiam (chave repetida), 0 sem data valida | importado em: 2026-07-22T13:54:02
[6/21] A processar: crono_1784656070233.xls
  -> 0 novas em 2025, 0 novas em 2026, 1265 ja existiam (chave repetida), 0 sem data valida | im